# Demand Forecasting Prep Notebook (from `model_v26.ipynb` + `train.csv`)

Notebook này chuẩn hóa dữ liệu đầu vào và xử lý lỗi kiểu dữ liệu số đang ở dạng `string`.


In [2]:
import pandas as pd
import numpy as np

# pd.set_option('display.max_columns', None)
# print('pandas:', pd.__version__)


In [3]:
# Load raw data
train = pd.read_csv('train.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print('train shape:', train.shape)
print('sample_submission shape:', sample_sub.shape)
print('Raw dtypes:')
print(train.dtypes)
train.head(3)


train shape: (711980, 8)
sample_submission shape: (31944, 29)
Raw dtypes:
Date              str
Stt            object
ItemCode          str
Quantity        int64
UnitPrice         str
SalesAmount     int64
Unit Cost         str
Cost Amount       str
dtype: object


/tmp/ipykernel_319056/2722786016.py:2: DtypeWarning: Columns (0: Stt) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('train.csv')


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700,2184300,"123559,1",1482709
1,2020-11-17,2000003,SKU-09458,600,"131818,1818",79090909,110000,66000000
2,2020-11-18,2000007,SKU-08062,6,230000,940909,101000,606000


## Fix lỗi kiểu dữ liệu (`string` -> `float`)

`UnitPrice` và `Unit Cost` có thể chứa số thập phân theo định dạng dấu phẩy (`131818,1818`).
Cell dưới sẽ chuẩn hóa về `float`.


In [5]:
def parse_decimal_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(' ', '')
    if s == '':
        return np.nan
    # Dataset này dùng dấu phẩy cho phần thập phân
    s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

numeric_string_cols = ['UnitPrice', 'Unit Cost']
for col in numeric_string_cols:
    train[col] = train[col].apply(parse_decimal_str).astype(float)

# Các cột int giữ lại dạng số
for col in ['Quantity', 'SalesAmount', 'Cost Amount', 'Stt']:
    train[col] = pd.to_numeric(train[col], errors='coerce')

train['Date'] = pd.to_datetime(train['Date'], errors='coerce')

print('Dtypes after casting:')
print(train.dtypes)
print('Null counts after casting:')
print(train[['Date', 'UnitPrice', 'Unit Cost', 'Quantity']].isna().sum())


Dtypes after casting:
Date           datetime64[us]
Stt                   float64
ItemCode                  str
Quantity                int64
UnitPrice             float64
SalesAmount             int64
Unit Cost             float64
Cost Amount           float64
dtype: object
Null counts after casting:
Date         0
UnitPrice    0
Unit Cost    0
Quantity     0
dtype: int64


In [6]:
# Optional sanity checks
# Recompute expected line totals from parsed floats and compare with provided integer totals.
train['SalesAmount_recalc'] = train['UnitPrice'] * train['Quantity']
train['CostAmount_recalc'] = train['Unit Cost'] * train['Quantity']

sales_abs_err = (train['SalesAmount_recalc'] - train['SalesAmount']).abs()
cost_abs_err = (train['CostAmount_recalc'] - train['Cost Amount']).abs()

print('Median abs error SalesAmount:', float(sales_abs_err.median()))
print('Median abs error Cost Amount:', float(cost_abs_err.median()))
print('95th pct abs error SalesAmount:', float(sales_abs_err.quantile(0.95)))
print('95th pct abs error Cost Amount:', float(cost_abs_err.quantile(0.95)))


Median abs error SalesAmount: 0.0
Median abs error Cost Amount: 0.24000000022351742
95th pct abs error SalesAmount: 603273.2400000002
95th pct abs error Cost Amount: 48372.76


## Tạo daily SKU frame cho mô hình

Cell này tạo dữ liệu mức ngày/SKU, net theo return (quantity âm).


In [7]:
daily = (
    train.groupby(['Date', 'ItemCode'], as_index=False)
         .agg(
             Quantity=('Quantity', 'sum'),
             SalesAmount=('SalesAmount', 'sum'),
             CostAmount=('Cost Amount', 'sum')
         )
         .sort_values(['ItemCode', 'Date'])
)

print('daily shape:', daily.shape)
daily.head(10)


daily shape: (507050, 5)


,Date,ItemCode,Quantity,SalesAmount,CostAmount
475989,2025-05-26,SKU-00001,1,830368,0.0
476372,2025-05-27,SKU-00001,1,1032840,0.0
476722,2025-05-28,SKU-00001,2,2084484,0.0
477421,2025-05-30,SKU-00001,2,1414600,0.0
478928,2025-06-04,SKU-00001,6,4138320,0.0
479317,2025-06-05,SKU-00001,1,1721000,0.0
481196,2025-06-11,SKU-00001,2,7096320,0.0
481944,2025-06-13,SKU-00001,2,5022040,0.0
485407,2025-06-26,SKU-00001,2,2911900,0.0
489526,2025-07-09,SKU-00001,1,373152,0.0


In [8]:
# Build a clean export (optional)
clean_path = 'train_clean.csv'
train.to_csv(clean_path, index=False)
print(f'Saved cleaned file: {clean_path}')


Saved cleaned file: train_clean.csv


## Gợi ý bước tiếp theo
- Dùng `train_clean.csv` để train pipeline trong `train_model.py`.
- Nếu muốn, có thể thêm cell chạy trực tiếp `train_model.py` bằng `!uv run python train_model.py ...`.


In [1]:
# Kiểm tra các lỗi dữ liệu phổ biến trong train_clean.csv
import pandas as pd
import numpy as np

df = pd.read_csv('train_clean.csv')

# 1. Quantity > 0 mà UnitPrice = 0
err1 = df[(df['Quantity'] > 0) & (df['UnitPrice'] == 0)]
print(f'Lỗi 1: Quantity > 0 nhưng UnitPrice = 0: {len(err1)} dòng')
display(err1.head())

# 2. UnitPrice âm
err2 = df[df['UnitPrice'] < 0]
print(f'Lỗi 2: UnitPrice âm: {len(err2)} dòng')
display(err2.head())

# 3. Q*P khác SalesAmount chênh lệch quá 1%
err3 = df[np.abs(df['Quantity'] * df['UnitPrice'] - df['SalesAmount']) > 0.01 * df['SalesAmount'].abs()]
print(f'Lỗi 3: Q*P khác SalesAmount >1%: {len(err3)} dòng')
display(err3.head())

# 4. UnitPrice lỗi nhập liệu (ví dụ quá lớn, quá nhỏ, hoặc NaN)
err4 = df[(df['UnitPrice'].isna()) | (df['UnitPrice'] < 1000) | (df['UnitPrice'] > 1e8)]
print(f'Lỗi 4: UnitPrice lỗi nhập liệu: {len(err4)} dòng')
display(err4.head())

Lỗi 1: Quantity > 0 nhưng UnitPrice = 0: 10461 dòng


,Date,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
236,2021-03-12,SKU-03479,29.0,0.0,0.0,15000.0,435000.0
271,2021-03-18,SKU-03479,60.0,0.0,0.0,15000.0,900000.0
274,2021-03-18,SKU-03479,6.0,0.0,0.0,15000.0,90000.0
275,2021-03-18,SKU-03479,6.0,0.0,0.0,15000.0,90000.0
276,2021-03-18,SKU-03479,6.0,0.0,0.0,15000.0,90000.0


Lỗi 2: UnitPrice âm: 37315 dòng


,Date,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
47,2020-12-09,SKU-08063,-12.0,-209102.25,-2509227.0,0.00,-1482709.0
48,2020-12-09,SKU-08061,-12.0,-231818.17,-2781818.0,0.00,-1897349.0
1185,2021-05-21,SKU-07770,-18.0,-231818.17,-4172727.0,-174000.00,-3132000.0
1317,2021-06-08,SKU-02768,-24.0,-161000.00,-3864000.0,-85000.00,-2040000.0
1321,2021-06-08,SKU-10532,-2.0,-676618.00,-1353236.0,-390065.25,-780131.0


Lỗi 3: Q*P khác SalesAmount >1%: 295159 dòng


,Date,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,SKU-08063,12.0,242700.00,2184300.0,123559.1,1482709.0
2,2020-11-18,SKU-08062,6.0,230000.00,940909.0,101000.0,606000.0
3,2020-11-18,SKU-09458,240.0,270000.00,44181816.0,110000.0,26400000.0
4,2020-11-18,SKU-09458,240.0,270000.00,44181816.0,110000.0,26400000.0
5,2020-11-20,SKU-09458,240.0,245454.55,44181816.0,110000.0,26400000.0


Lỗi 4: UnitPrice lỗi nhập liệu: 51049 dòng


,Date,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
47,2020-12-09,SKU-08063,-12.0,-209102.25,-2509227.0,0.0,-1482709.0
48,2020-12-09,SKU-08061,-12.0,-231818.17,-2781818.0,0.0,-1897349.0
78,2020-12-30,SKU-07770,0.0,0.00,0.0,174000.0,0.0
236,2021-03-12,SKU-03479,29.0,0.00,0.0,15000.0,435000.0
271,2021-03-18,SKU-03479,60.0,0.00,0.0,15000.0,900000.0
